# Demand Forecasting & Inventory Optimization Engine (PySpark)

Personal project on the public **Kaggle Superstore Sales** dataset (9,800 order lines, 2015–2018).
The pipeline turns raw transactions into **weekly sales forecasts per product sub-category**, compares them with simple baselines, and converts forecast error into **safety buffers at a 95% service level**.

> All values are **sales dollars**, not units: the dataset has no quantity column.

# 1. Environment Setup

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Point JAVA_HOME at the Colab JDK
    try:
        java_path = os.path.realpath(subprocess.check_output(["which", "java"]).decode().strip())
        os.environ["JAVA_HOME"] = os.path.dirname(os.path.dirname(java_path))
    except Exception as e:
        print(f"Failed to detect Java: {e}")
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    !pip install pyspark==4.0.1 -q
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("SupplyChain_Forecast")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")

Spark version: 4.0.1


# 2. Data Ingestion

Product names contain inch marks written as doubled quotes (e.g. `14 7/8"" x 11""`). Spark's CSV reader escapes with a backslash by default, which **shifts columns on those rows**. Setting `escape='"'` reads the file as the standard CSV it is.

In [2]:
# NOTE: update DATA_PATH to point at your copy of train.csv
DATA_PATH = ("/content/drive/MyDrive/Colab Notebooks/Data/train.csv" if IN_COLAB
             else os.environ.get("SUPERSTORE_CSV", "train.csv"))

df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .option("escape", '"')          # RFC 4180: "" inside a quoted field = one literal quote
      .csv(DATA_PATH))

print(f"Total records: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
df.printSchema()

Total records: 9,800
Total columns: 18
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)



# 3. Data Cleaning & Quality Check

In [3]:
from pyspark.sql import functions as F

df_clean = (df
            .withColumn("Sales", F.col("Sales").cast("double"))
            .withColumn("Order Date", F.to_date("Order Date", "dd/MM/yyyy")))  # day-first dates

total = df_clean.count()
bad_sales = df_clean.filter(F.col("Sales").isNull()).count()
bad_dates = df_clean.filter(F.col("Order Date").isNull()).count()

df_final = df_clean.dropna(subset=["Sales", "Order Date"])

print("--- Data Quality Report ---")
print(f"Rows read:            {total:,}")
print(f"Invalid sales values: {bad_sales}")
print(f"Invalid order dates:  {bad_dates}")
print(f"Rows kept:            {df_final.count():,}")
print(f"Order date range:     {df_final.agg(F.min('Order Date')).first()[0]} to {df_final.agg(F.max('Order Date')).first()[0]}")

--- Data Quality Report ---
Rows read:            9,800
Invalid sales values: 0
Invalid order dates:  0
Rows kept:            9,800
Order date range:     2015-01-03 to 2018-12-30


# 4. Weekly Aggregation (complete weekly grid)

Order lines are summed to **weekly sales per sub-category** (weeks start on Monday). Weeks with no sales are **filled with 0**, so a 1-week lag always means the previous calendar week.

In [4]:
weekly = (df_final
          .withColumn("Week_Start", F.date_trunc("week", "Order Date").cast("date"))
          .groupBy("Week_Start", "Sub-Category")
          .agg(F.sum("Sales").alias("Weekly_Sales")))

first_week, last_week = weekly.agg(F.min("Week_Start"), F.max("Week_Start")).first()
all_weeks = spark.sql(
    f"SELECT explode(sequence(DATE'{first_week}', DATE'{last_week}', INTERVAL 1 WEEK)) AS Week_Start")
sub_categories = weekly.select("Sub-Category").distinct()

df_weekly = (all_weeks.crossJoin(sub_categories)
             .join(weekly, ["Week_Start", "Sub-Category"], "left")
             .fillna(0.0, subset=["Weekly_Sales"])
             .withColumn("Week_of_Year", F.weekofyear("Week_Start")))

n_weeks = all_weeks.count()
n_subs = sub_categories.count()
print(f"{n_weeks} weeks x {n_subs} sub-categories = {df_weekly.count():,} weekly rows")
print(f"Zero-sales weeks filled: {df_weekly.filter(F.col('Weekly_Sales') == 0).count():,}")

209 weeks x 17 sub-categories = 3,553 weekly rows
Zero-sales weeks filled: 867


# 5. Feature Engineering

All features use **only past weeks**: the rolling average covers the 4 weeks *before* the target week (`rowsBetween(-4, -1)`), so the model never sees the sales it is predicting.

In [5]:
from pyspark.sql.window import Window

w = Window.partitionBy("Sub-Category").orderBy("Week_Start")

df_features = (df_weekly
               .withColumn("Lag_1_Week", F.lag("Weekly_Sales", 1).over(w))
               .withColumn("Lag_4_Weeks", F.lag("Weekly_Sales", 4).over(w))
               .withColumn("Rolling_Avg_Prev_4_Weeks", F.avg("Weekly_Sales").over(w.rowsBetween(-4, -1))))

df_model_data = df_features.dropna()   # drops each sub-category's first 4 weeks

df_model_data.orderBy("Sub-Category", "Week_Start").show(5)

+----------+------------+------------------+------------+------------------+-----------+------------------------+
|Week_Start|Sub-Category|      Weekly_Sales|Week_of_Year|        Lag_1_Week|Lag_4_Weeks|Rolling_Avg_Prev_4_Weeks|
+----------+------------+------------------+------------+------------------+-----------+------------------------+
|2015-01-26| Accessories|             468.9|           5|               0.0|        0.0|      206.97250000000003|
|2015-02-02| Accessories|            115.36|           6|             468.9|       31.2|                324.1975|
|2015-02-09| Accessories|474.41999999999996|           7|            115.36|     796.69|                345.2375|
|2015-02-16| Accessories|             62.31|           8|474.41999999999996|        0.0|      264.66999999999996|
|2015-02-23| Accessories|               0.0|           9|             62.31|      468.9|      280.24749999999995|
+----------+------------+------------------+------------+------------------+-----------+

# 6. Modeling

Time-based split: train on weeks before **2017-01-02**, test on 2017–2018. A fixed seed and a fixed row order make the result reproducible.

In [6]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

FEATURES = ["Week_of_Year", "Lag_1_Week", "Lag_4_Weeks", "Rolling_Avg_Prev_4_Weeks"]
SPLIT_DATE = "2017-01-02"

df_prep = VectorAssembler(inputCols=FEATURES, outputCol="features").transform(df_model_data)

train = df_prep.filter(F.col("Week_Start") < SPLIT_DATE).orderBy("Sub-Category", "Week_Start").coalesce(1)
test = df_prep.filter(F.col("Week_Start") >= SPLIT_DATE)

print(f"Training rows (before {SPLIT_DATE}): {train.count():,}")
print(f"Test rows ({SPLIT_DATE} onward):     {test.count():,}")

rf = RandomForestRegressor(featuresCol="features", labelCol="Weekly_Sales", numTrees=100, seed=42)
model = rf.fit(train)
predictions = model.transform(test)

Training rows (before 2017-01-02): 1,717
Test rows (2017-01-02 onward):     1,768


# 7. Evaluation vs. Baselines

A forecast is only useful if it beats simple rules. Two baselines:
- **Naive:** next week = last week's sales
- **4-week moving average:** next week = average of the previous 4 weeks

In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

def score(pred_col):
    rmse = RegressionEvaluator(labelCol="Weekly_Sales", predictionCol=pred_col, metricName="rmse").evaluate(predictions)
    mae = RegressionEvaluator(labelCol="Weekly_Sales", predictionCol=pred_col, metricName="mae").evaluate(predictions)
    return rmse, mae

results = {
    "Random Forest": score("prediction"),
    "Naive (last week)": score("Lag_1_Week"),
    "4-week moving average": score("Rolling_Avg_Prev_4_Weeks"),
}

rf_rmse, rf_mae = results["Random Forest"]
print(f"{'Model':<24}{'RMSE ($)':>10}{'MAE ($)':>10}{'RF RMSE gain':>14}")
for name, (rmse, mae) in results.items():
    gain = "" if name == "Random Forest" else f"{(1 - rf_rmse / rmse):.1%}"
    print(f"{name:<24}{rmse:>10,.0f}{mae:>10,.0f}{gain:>14}")

Model                     RMSE ($)   MAE ($)  RF RMSE gain
Random Forest                1,310       641              
Naive (last week)            1,772       822         26.1%
4-week moving average        1,424       719          8.0%


# 8. Inventory Optimization

Each sub-category's forecast error (RMSE) sets its safety buffer:
**Safety Buffer = Z × RMSE**, with Z = 1.65 for a 95% service level.
Volatile sub-categories get larger buffers; stable ones get smaller ones. Values are in sales dollars.

In [8]:
Z_SCORE = 1.65   # 95% service level

sub_rmse = (predictions
            .groupBy("Sub-Category")
            .agg(F.sqrt(F.mean((F.col("Weekly_Sales") - F.col("prediction")) ** 2)).alias("Forecast_RMSE"))
            .withColumn("Safety_Buffer", F.round(F.col("Forecast_RMSE") * Z_SCORE, 0)))

print("Safety buffer by sub-category ($ per week):")
sub_rmse.orderBy(F.desc("Safety_Buffer")).select("Sub-Category", F.round("Forecast_RMSE", 0).alias("Forecast_RMSE"), "Safety_Buffer").show(17)

Safety buffer by sub-category ($ per week):
+------------+-------------+-------------+
|Sub-Category|Forecast_RMSE|Safety_Buffer|
+------------+-------------+-------------+
|     Copiers|       3226.0|       5322.0|
|    Machines|       2110.0|       3481.0|
|      Chairs|       1693.0|       2794.0|
|     Binders|       1605.0|       2648.0|
|      Phones|       1576.0|       2601.0|
|      Tables|       1214.0|       2004.0|
|     Storage|       1148.0|       1895.0|
|    Supplies|        918.0|       1515.0|
| Accessories|        894.0|       1475.0|
|   Bookcases|        804.0|       1327.0|
|  Appliances|        774.0|       1277.0|
| Furnishings|        605.0|        999.0|
|       Paper|        407.0|        671.0|
|         Art|        230.0|        379.0|
|      Labels|        203.0|        336.0|
|   Envelopes|        175.0|        289.0|
|   Fasteners|        166.0|        274.0|
+------------+-------------+-------------+



In [9]:
plan = (predictions.join(sub_rmse, "Sub-Category")
        .withColumn("Forecast_Sales", F.round("prediction", 0))
        .withColumn("Total_Requirement", F.col("Forecast_Sales") + F.col("Safety_Buffer")))

print("=== Weekly replenishment plan: highest requirements ($) ===")
(plan.select("Week_Start", "Sub-Category", "Forecast_Sales", "Safety_Buffer", "Total_Requirement")
     .orderBy(F.desc("Total_Requirement"), "Week_Start")
     .show(10))

=== Weekly replenishment plan: highest requirements ($) ===
+----------+------------+--------------+-------------+-----------------+
|Week_Start|Sub-Category|Forecast_Sales|Safety_Buffer|Total_Requirement|
+----------+------------+--------------+-------------+-----------------+
|2018-11-19|     Copiers|        2174.0|       5322.0|           7496.0|
|2018-10-29|     Copiers|        1953.0|       5322.0|           7275.0|
|2018-12-10|     Copiers|        1825.0|       5322.0|           7147.0|
|2017-10-09|     Copiers|        1680.0|       5322.0|           7002.0|
|2018-11-05|     Copiers|        1627.0|       5322.0|           6949.0|
|2017-09-11|     Copiers|        1442.0|       5322.0|           6764.0|
|2017-10-30|     Copiers|        1314.0|       5322.0|           6636.0|
|2018-11-26|     Copiers|        1313.0|       5322.0|           6635.0|
|2018-04-02|     Copiers|        1180.0|       5322.0|           6502.0|
|2018-12-24|     Copiers|        1140.0|       5322.0|          

# 9. Limitations
- Values are sales dollars, not units (the dataset has no quantity column).
- Safety buffers are sized from test-period error; a production setup would size them from a rolling validation window and add supplier lead time.
- One pooled model across all sub-categories; features are lags and calendar week only (no price, promotion or holiday data).